In [7]:
import os
import cv2

# ========================= CONFIGURATION =========================
# Point this to your folder containing the gas meter images
IMAGE_DIR = r"..\data_Set\\train_2\\images" 
# =================================================================

def main():
    if not os.path.exists(IMAGE_DIR):
        print(f"Error: The directory '{IMAGE_DIR}' does not exist. Check your path.")
        return

    # Automatically set up the labels directory parallel to the images folder
    # Structure created: 
    # ..\data_Set\train_2\images (your images directory)
    # ..\data_Set\train_2\labels (your labels directory)
    parent_dir = os.path.dirname(IMAGE_DIR)
    labels_dir = os.path.join(parent_dir, "labels")
    
    # Create the labels directory if it doesn't exist yet
    os.makedirs(labels_dir, exist_ok=True)
    print(f"Labels directory established at: {labels_dir}")

    # Supported image extensions
    valid_exts = (".png", ".jpg", ".jpeg", ".bmp")
    images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_exts)])

    if not images:
        print(f"No images found in '{IMAGE_DIR}'.")
        return

    print("\n--- INSTRUCTIONS ---")
    print("1. Drag a box around the digit strip with your mouse.")
    print("2. Press ENTER or SPACE to save the coordinates and move to the next image.")
    print("3. Press 'c' to clear the current box and drag again.")
    print("4. Press 'd' on your keyboard to DELETE the image and its saved label.")
    print("5. Press 'Esc' key to EXIT.")
    print("-" * 25 + "\n")

    idx = 0
    user_terminated = False
    
    while idx < len(images):
        img_name = images[idx]
        img_path = os.path.join(IMAGE_DIR, img_name)
        
        # Save the .txt file with the exact same name as the image but inside the 'labels' directory
        base_no_ext = os.path.splitext(img_name)[0]
        txt_path = os.path.join(labels_dir, f"{base_no_ext}.txt")

        # Skip if we already annotated this image (allows resuming later)
        if os.path.exists(txt_path):
            idx += 1
            continue

        # Double check the image file still exists on disk
        if not os.path.exists(img_path):
            idx += 1
            continue

        frame = cv2.imread(img_path)
        if frame is None:
            print(f"Skipping unreadable image: {img_name}")
            idx += 1
            continue

        display_frame = frame.copy()
        print(f"[{idx + 1}/{len(images)}] Annotating: {img_name}")

        # Open the ROI selector window
        window_title = f"Crop: {img_name} (Esc to Quit | d to Delete)"
        roi = cv2.selectROI(window_title, display_frame, showCrosshair=True, fromCenter=False)
        
        # Force the window to destroy and release UI focus immediately after drawing/interaction
        cv2.destroyWindow(window_title)
        cv2.destroyAllWindows() 

        x, y, w, h = roi

        # If window was closed or key was pressed without drawing a box
        if w == 0 or h == 0:
            key = cv2.waitKey(1) & 0xFF
            
            # Action: Delete Image & Matching Label
            if key == ord('d'):
                confirm = input(f"Are you sure you want to DELETE {img_name} from disk? (y/n): ").lower().strip()
                if confirm == 'y':
                    try:
                        os.remove(img_path)
                        if os.path.exists(txt_path):
                            os.remove(txt_path)
                        print(f"Deleted image and clean-up complete.\n")
                    except Exception as e:
                        print(f"Failed to delete: {e}\n")
                else:
                    print("Delete canceled.\n")
                idx += 1
                continue

            # Action: Esc / Exit
            else:
                user_quit = input("Do you want to exit? (y/n): ").lower().strip()
                if user_quit == 'y':
                    user_terminated = True
                    break
                else:
                    continue

        # Calculate bounding box coordinates
        xmin = int(x)
        ymin = int(y)
        xmax = int(x + w)
        ymax = int(y + h)

        # Save coordinates to the dedicated labels directory
        try:
            with open(txt_path, 'w') as f:
                f.write(f"{xmin} {ymin} {xmax} {ymax}\n")
            print(f"Saved coordinates to: labels/{os.path.basename(txt_path)}\n")
        except Exception as e:
            print(f"Error saving label file: {e}\n")

        idx += 1

    # Cleanup any leftover OpenCV windows
    cv2.destroyAllWindows()

    # ========================= COMPLETENESS CHECK =========================
    print("\n--- DATASET COMPLETENESS CHECK ---")
    # Fetch existing files again to verify completeness
    remaining_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_exts)])
    unlabeled_count = 0
    
    for img in remaining_images:
        base = os.path.splitext(img)[0]
        corresponding_lbl = os.path.join(labels_dir, f"{base}.txt")
        if not os.path.exists(corresponding_lbl):
            unlabeled_count += 1
            print(f"MISSING LABEL: {img}")

    if unlabeled_count == 0:
        print("Success! Every single image currently in your dataset has been labeled.")
    else:
        print(f"Attention: There are still {unlabeled_count} image(s) left without a label file.")
        if user_terminated:
            print("Run the script again; it will automatically skip annotated images and pick up where you left off.")


if __name__ == "__main__":
    main()

Labels directory established at: ..\data_Set\\train_2\labels

--- INSTRUCTIONS ---
1. Drag a box around the digit strip with your mouse.
2. Press ENTER or SPACE to save the coordinates and move to the next image.
3. Press 'c' to clear the current box and drag again.
4. Press 'd' on your keyboard to DELETE the image and its saved label.
5. Press 'Esc' key to EXIT.
-------------------------

[8/626] Annotating: 14710588774.jpg
Saved coordinates to: labels/14710588774.txt

[9/626] Annotating: 14751622417.jpg
Saved coordinates to: labels/14751622417.txt

[10/626] Annotating: 15088923501.jpg
Saved coordinates to: labels/15088923501.txt

[11/626] Annotating: 15095173009.jpg

--- DATASET COMPLETENESS CHECK ---
MISSING LABEL: 15095173009.jpg
MISSING LABEL: 15180010009.jpg
MISSING LABEL: 15214394643.jpg
MISSING LABEL: 15242568432.jpg
MISSING LABEL: 15496526656.jpg
MISSING LABEL: 15695753689.jpg
MISSING LABEL: 15753911120.jpg
MISSING LABEL: 15807051519.jpg
MISSING LABEL: 15856736846.jpg
MISSING 